In [1]:
%pip install torch torchvision transformers pillow open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
  

In [6]:
import torch
import numpy as np
from PIL import Image
import open_clip
from transformers import BeitImageProcessor, BeitModel
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('hf-hub:apple/DFN5B-CLIP-ViT-H-14', device=device)

clip_model.eval()

def get_clip_embedding(image: Image.Image):
    # image: PIL Image RGB
    with torch.no_grad():
        image_tensor = clip_preprocess(image).unsqueeze(0).to(device)
        emb = clip_model.encode_image(image_tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.cpu().numpy()[0]  # 1D numpy vector

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_pytorch_model.bin:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

open_clip_config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

In [8]:
beit_processor = BeitImageProcessor.from_pretrained("microsoft/beit-base-patch16-224")
beit_model = BeitModel.from_pretrained("microsoft/beit-base-patch16-224").to(device)

beit_model.eval()

def get_beit_embedding(image: Image.Image):
    inputs = beit_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = beit_model(**inputs)
        cls_emb = outputs.last_hidden_state[:, 0]  # Lấy embedding của token [CLS]
        cls_emb = cls_emb / cls_emb.norm(dim=-1, keepdim=True)  # Chuẩn hóa
        return cls_emb.cpu().numpy()[0]


/usr/local/lib/python3.11/dist-packages/transformers/utils/deprecation.py:172: UserWarning: The following named arguments are not valid for `BeitImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)
